In [10]:
print(ord('牛'))
print(chr(29275))
print(hex(29275))
print(bin(29275))
print('牛'.encode('utf-8'))

29275
牛
0x725b
0b111001001011011
b'\xe7\x89\x9b'


In [28]:
import regex as re
from collections import Counter, defaultdict
from rich import print as pprint

In [65]:
import regex as re  # 注意：必须使用 regex 库以支持 \p{L} 等高级 Unicode 属性
from collections import Counter, defaultdict

def bbpe(input_path, output_path, vocab_size, special_tokens=["<|endoftext|>"]):
    # --- 1. 初始化词表 (0-255 的基础字节) ---
    vocab = {idx: bytes([idx]) for idx in range(256)}

    # 读取输入文本
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # --- 2. 预处理：保护特殊 Token 并进行预分词 ---
    if special_tokens:
        # 将特殊 Token 转义并合并为正则串，用于切割文本
        sp_regex = '|'.join(re.escape(token) for token in special_tokens)
        parts = re.split(f'({sp_regex})', text)
        # 仅保留非特殊 Token 的文本片段用于训练
        train_segments = [s for s in parts if s and s not in special_tokens]
    else:
        train_segments = [text]
    
    # GPT-2 的官方预分词正则：处理缩写、单词、数字及空格
    gpt2_pattern = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+")
    
    raw_counts = Counter()
    for segment in train_segments:
        words = gpt2_pattern.findall(segment)
        for word in words:
            # 将单词转为 UTF-8 字节流，并拆解为单字节元组作为初始单位
            # 例如: "hi" -> (b'h', b'i')
            raw_counts[tuple(bytes([b]) for b in word.encode('utf-8'))] += 1
    
    # 为了方便索引，将 Counter 展开为列表形式
    words_list, counts_list = [], []
    for word_tuple, freq in raw_counts.items():
        words_list.append(list(word_tuple)) # 使用 list 方便后续原地修改
        counts_list.append(freq)
    
    # --- 3. 构造高效统计数据结构 ---
    stats = defaultdict(int)      # 存储相邻对 (pair) 的全局出现频率
    indices = defaultdict(set)    # 倒排索引：存储 pair 出现在 words_list 中的哪些位置 (idx)
    
    for idx, word in enumerate(words_list):
        freq = counts_list[idx]
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            stats[pair] += freq
            indices[pair].add(idx)
    
    # --- 4. 核心合并循环 ---
    # 计算需要进行的合并次数
    num_merges = vocab_size - len(vocab) - len(special_tokens)
    merges = []
    
    for _ in range(num_merges):
        if not stats:
            break
        
        # 寻找当前频率最高且字典序最大的字节对
        best_pair = max(stats.items(), key=lambda x: (x[1], x[0]))[0]
        
        # 异常处理：如果最高频率都小于0（逻辑理论不应发生），则停止
        if stats[best_pair] < 0:
            break
        
        merges.append(best_pair)
        new_token = best_pair[0] + best_pair[1] # 拼接字节对形成新子词
        
        # 仅针对包含 best_pair 的单词进行更新（利用倒排索引加速）
        relevant_indices = list(indices[best_pair])
        
        for idx in relevant_indices:
            word = words_list[idx]
            freq = counts_list[idx]
            
            i = 0
            while i < len(word) - 1:
                # 寻找匹配 best_pair 的位置
                if word[i] == best_pair[0] and word[i + 1] == best_pair[1]:
                    # --- A. 更新被破坏的旧邻居对频率 ---
                    # 处理左侧邻居：(word[i-1], word[i]) 将不再存在
                    if i > 0:
                        prev_pair = (word[i - 1], word[i])
                        stats[prev_pair] -= freq
                        if stats[prev_pair] <= 0: # 频率为0时清理，优化下一次 max()
                            del stats[prev_pair]
                    
                    # 处理右侧邻居：(word[i+1], word[i+2]) 将不再存在
                    if i < len(word) - 2:
                        next_pair = (word[i + 1], word[i + 2])
                        stats[next_pair] -= freq
                        if stats[next_pair] <= 0:
                            del stats[next_pair]
                    
                    # --- B. 执行原地合并 ---
                    word[i] = new_token
                    del word[i + 1]
                    
                    # --- C. 更新产生的新邻居对频率及索引 ---
                    # 处理新的左邻居：(word[i-1], new_token)
                    if i > 0:
                        new_prev = (word[i - 1], word[i])
                        stats[new_prev] += freq
                        indices[new_prev].add(idx)
                    
                    # 处理新的右邻居：(new_token, word[i+1])
                    if i < len(word) - 1:
                        new_next = (word[i], word[i + 1])
                        stats[new_next] += freq
                        indices[new_next].add(idx)
                    
                    # 合并成功后，i 不需要自增，因为列表长度缩短，word[i] 已经是新元素
                else:
                    i += 1
        
        # 完成该 pair 在全语料的合并后，从排行和索引中彻底清理
        if best_pair in stats:
            del stats[best_pair]
        if best_pair in indices:
            del indices[best_pair]
    
    # --- 5. 构建最终词表 ---
    # 将合并过程中产生的子词按顺序加入词表
    for pair in merges:
        new_token = pair[0] + pair[1]
        vocab[len(vocab)] = new_token

    # 将特殊 Token 加入词表末尾
    for token in special_tokens:
        vocab[len(vocab)] = token.encode('utf-8')
    
    return vocab, merges

In [72]:
# input_path = "data/TinyStoriesV2-GPT4-train.txt"
input_path = "data/test.txt"
output_path = ""

vocab, merges = bbpe(input_path, output_path, 10000)
pprint("Final Vocabulary Size:", len(vocab))

Final Vocabulary Size: 761